In [1]:
# 测试 unsloth
import unsloth
print(unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/miniconda3/envs/rasa-finetuning/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
2026.2.1


In [3]:
# 加载并量化基础模型
from unsloth import FastLanguageModel
from transformers import BitsAndBytesConfig

random_seed = 42
max_seq_length = 2048

# 量化方式
# quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# 加载量化模型和分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/model/ModelScope/Qwen/Qwen3-8B",
    max_seq_length=max_seq_length,
    load_in_16bit = True
)

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 5/5 [00:13<00:00,  2.68s/it]


In [4]:
# 配置模型以进行PEFT(Parameter Efficient Fine-Tuning，参数高效微调)
# from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16, # LoRA 的秩（rank）
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ], # 指定要应用 LoRA 的模块名称
    lora_alpha=16, # LoRA 中的缩放系数
    lora_dropout=0, # LoRA 部分的 dropout 概率
    bias="none", # 不微调 bias 参数
    use_gradient_checkpointing="unsloth", # 启用梯度检查点以节省显存
    random_state=random_seed, # LoRA 初始化的随机种子
    use_rslora=False, # 不启用 Rank Stabilized LoRA (RS-LoRA)
    loftq_config=None, # 不配置 LoFTQ
)

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [5]:
# 加载训练和验证数据集
import datasets
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

# 加载数据集
train_dataset = datasets.load_dataset("json", data_files={"train": "ft_splits/train.jsonl"}, split="train")
eval_dataset = datasets.load_dataset("json", data_files={"eval": "ft_splits/val.jsonl"}, split="eval")

# 使用对话模板格式化数据
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
def formatting_func(example):
    return {
        "text": f"<|im_start|>user\n{example['prompt'].strip()}<|im_end|>\n"
                f"<|im_start|>assistant\n{example['completion'].strip()}<|im_end|>"
    }
train_dataset = train_dataset.map(formatting_func)
eval_dataset  = eval_dataset.map(formatting_func)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


In [6]:
# 配置训练器
# 配置参数
args = TrainingArguments(
    ###### 训练参数
    seed = random_seed,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    ###### 数据类型
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    ###### 验证参数
    eval_strategy = "steps",
    eval_steps = 4,
    per_device_eval_batch_size = 1,
    ###### 输出参数
    logging_steps = 1,
    output_dir = "outputs",
)

# 使用参数配置训练器
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    max_seq_length = max_seq_length,
    args = args,
)

In [7]:
# 微调
finetune_metrics = trainer.train()

# 以16bit精度保存模型
# model.save_pretrained_merged("/root/finetuned_model/Qwen3-8B", tokenizer, save_method="merged_16bit")

from peft import PeftModel
# 保存 adapter
model.save_pretrained("/root/finetuned_model/Qwen3-8B-adapter-only")
tokenizer.save_pretrained("/root/finetuned_model/Qwen3-8B-adapter-only")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 88 | Num Epochs = 2 | Total steps = 88
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Step,Training Loss,Validation Loss
4,2.505900,2.411136
8,2.012300,1.952729
12,1.690400,1.610093
16,1.290700,1.213010
20,0.925900,0.763308
24,0.462800,0.327168
28,0.102800,0.164689
32,0.057400,0.133077
36,0.175000,0.117901
40,0.134400,0.100117


('/root/finetuned_model/Qwen3-8B-adapter-only/tokenizer_config.json',
 '/root/finetuned_model/Qwen3-8B-adapter-only/special_tokens_map.json',
 '/root/finetuned_model/Qwen3-8B-adapter-only/chat_template.jinja',
 '/root/finetuned_model/Qwen3-8B-adapter-only/vocab.json',
 '/root/finetuned_model/Qwen3-8B-adapter-only/merges.txt',
 '/root/finetuned_model/Qwen3-8B-adapter-only/added_tokens.json',
 '/root/finetuned_model/Qwen3-8B-adapter-only/tokenizer.json')

In [8]:
# 推理，加载微调后的模型
from transformers import TextStreamer
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained("/model/ModelScope/Qwen/Qwen3-8B")
model.load_adapter("/root/finetuned_model/Qwen3-8B-adapter-only")
FastLanguageModel.for_inference(model)  # 开启推理优化

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 5/5 [00:11<00:00,  2.23s/it]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151643)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
      

In [9]:
# 应用提示模版并分词
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": train_dataset["prompt"][19]}],  # 使用 TRL 对话格式
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# 根据用户输入生成模型输出
_ = model.generate(
    input_ids=input_ids,
    streamer=TextStreamer(tokenizer), # 生成时使用流式输出
    max_new_tokens=32768,
    do_sample=False, # 禁用随机采样，生成确定性输出
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
## Task Description
Your task is to analyze the current conversation context and generate a list of actions to start new business processes that we call flows, to extract slots, or respond to small talk and knowledge requests.

---

### Date & Time Context
- Current date: 02 March, 2026   (DD Month, YYYY)
- Current time: 08:44:09 (UTC)  (HH:MM:SS, 24-hour format with timezone)
- Current day: Monday     (Day of week)

---

## Available Flows and Slots
Use the following structured data:
```json
{"flows":[{"name":"query_logistics_companys","description":"查询支持的快递公司"},{"name":"query_shipping_order_logistics","description":"查询进行中订单的物流信息","slots":[{"name":"order_id"}]},{"name":"logistics_complaint","description":"投诉物流","slots":[{"name":"order_id"},{"name":"logistics_complaint"},{"name":"other_logistics_complaint","description":"详细的投诉内容"}]},{"name":"switch_user_id","description":"切换账号","slots":[{"name":"user_id","description":"用户id"}]},{"name":"query_order_detail","description

In [ ]:
# vllm serve /root/llms/Qwen/Qwen3-8B --served-model-name Qwen3-8B-sft --max-model-len 2048 --enable-lora --lora-modules alora=/root/finetuned_model/Qwen3-8B-adapter-only